# Gain
Current growth rate vs bias, impact ionization, I-V and spectra.
Figures are saved in `SAVE_DIR` as `.png`, `.pdf` and editable `.mplfig`
(reopen with `python open_fig.py <file>.mplfig` or `mcplot.load_figure`).
Plotting functions are in `transient_plots.py`.

In [ ]:
# Interactive figures (zoom/pan) need ipympl; otherwise fall back to static figures
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")
import os

import matplotlib.pyplot as plt
import numpy as np
import transient_plots as tp
from mcplot import save_figure, set_style

set_style(25)

SAVE_FIGURES = True  # save every figure as .png, .pdf and editable .mplfig
SAVE_DIR = "output/gain"


def save(fig, name):
    """Save fig as SAVE_DIR/name.{png,pdf,mplfig} (reopen the .mplfig with open_fig.py)."""
    if SAVE_FIGURES:
        save_figure(fig, os.path.join(SAVE_DIR, name), formats=("png", "pdf"), dpi=300, bbox_inches="tight")

# Current growth rate vs bias
Slope of a linear fit of $\log_{10}(I)$ vs t over all the positive current samples.

In [ ]:
base_path = "/mnt/csmhome2/amudano/MonteCarlo/DEMC/SPAD_FBK_v2/quasi_1D"
bias = np.linspace(-30, -37, 8)
folder_list = [f"FF/breakdown-{v:.2f}V" for v in bias]
contact = "pcontact"

In [ ]:
plt.close("all")
vdc_list, slope_list = [], []
for folder in folder_list:
    r = tp.read_transient(base_path, folder, contact)
    positive = r.current > 0
    if positive.sum() > 1:
        slope, _ = np.polyfit(r.t[positive], np.log10(r.current[positive]), 1)  # log10(I) = slope*t + b
        vdc_list.append(r.vdc)
        slope_list.append(slope)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(vdc_list, slope_list, "o-", markersize=10)
ax.set_xlabel("Bias Voltage (V)")
ax.set_ylabel("Slope of log$_{10}$(I) vs t (1/s)")
ax.set_title("Current growth rate (log scale) vs Bias Voltage")
ax.grid(True)
for v, s in zip(vdc_list, slope_list):
    ax.annotate(f"{s:.2e}", (v, s), textcoords="offset points", xytext=(5, 5), fontsize=10)
save(fig, "slope_vs_bias")
plt.show()

# Impact ionization
## Current and impact ionization events

In [ ]:
base_path = "/mnt/csmhome2/amudano/MonteCarlo/DEMC/SPAD_FBK_v2/quasi_2D"
folder_list = ["CONTINUE/1SC/quenching--37.00V"]
# folder_list = [f"FF/breakdown-1SC-{v:.2f}V" for v in [-37]]
label_list = ["Bias = -37V"]

In [ ]:
plt.close("all")
fig = tp.current_and_ionization_figure(base_path, folder_list, label_list, contact="ncontact", figsize=(15, 7))
save(fig, "current_and_ionization")
fig = tp.diode_and_capacitive_current_figure(base_path, folder_list, label_list, contact="ncontact",
                                             legend_loc="upper left")
save(fig, "diode_and_capacitive_current")
plt.show()

## Impact ionization events

In [ ]:
base_path = "/mnt/csmhome2/amudano/MonteCarlo/DEMC/SPAD_FBK_v2/quasi_1D"
bias = [-37, -33]
folder_list = [f"FF/breakdown-{v:.2f}V" for v in bias]
# folder_list = ["CONTINUE/quenching--37.00V"]
label_list = [f"{v}V" for v in bias]
t_stop = 26e-12  # histogram range: [0, t_stop]

In [ ]:
plt.close("all")
fig = tp.ionization_events_figure(base_path, folder_list, label_list, t_stop, figsize=(27, 7),
                                  show_bin_width=False, reverse_legend=True)
save(fig, "ionization_events")
fig = tp.diode_and_capacitive_current_figure(base_path, folder_list, label_list, contact="ncontact",
                                             legend_loc="upper left")
save(fig, "diode_and_capacitive_current_events")
plt.show()

# I-V characteristic
Mean of the last 10 current samples of every voltage-ramp simulation.

In [ ]:
set_style(15)
base_path = "/mnt/polmcad/MonteCarlo/DEMC/SPAD_FBK_v2/3D/MC"
bias = np.linspace(-35, -15, 21)

In [ ]:
plt.close("all")
fig = tp.iv_figure(base_path, bias, folder_fmt="voltage_ramp-{v:.2f}V", contact="pcontact")
save(fig, "iv_characteristic")
plt.show()

# Current transient

In [ ]:
base_path = "/mnt/csmhome2/amudano/MonteCarlo/DEMC/SPAD_FBK_v2/3D"
folder_list = ["MC/new_mesh/voltage_ramp--37.00V"]
# folder_list = [f"MC/new_mesh/voltage_ramp-1fs-{v:.2f}V" for v in [-37]]
label_list = ["1000SC", "1SC"]

In [ ]:
plt.close("all")
fig = tp.current_transient_figure(base_path, folder_list, label_list, contact="pcontact", figsize=(20, 7),
                                  log=False, legend_loc="upper right")
save(fig, "current_transient")
plt.show()

## Savitzky-Golay filtered current

In [ ]:
window_length = 51  # must be odd
polyorder = 3

In [ ]:
plt.close("all")
fig = tp.current_transient_figure(base_path, folder_list, label_list, contact="ncontact", figsize=(8, 6),
                                  log=True, legend_loc=None, sv_filter=(window_length, polyorder),
                                  label_fmt="Vbias = {vdc} ")
save(fig, "current_transient_sv_filter")
plt.show()

## Current spectrum (FFT)
Uses the folders of the *Current transient* section. Valid only for constant dt.

In [ ]:
plt.close("all")
fig = tp.current_fft_figure(base_path, folder_list, label_list, contact="p-contact")
save(fig, "current_fft")
plt.show()

# Scattering rate

In [ ]:
path = "/mnt/polmcad/MonteCarlo/Materials/silicon_mio/scattering/Energy_Rates"
scattering_file = f"{path}/e_acoabs_TA_band_5_band_5.txt"

In [ ]:
plt.close("all")
fig = tp.scattering_rate_figure(scattering_file, xlim=(0, 4))
save(fig, "scattering_rate")
plt.show()